<a href="https://www.kaggle.com/code/isithadinujaya/vision?scriptVersionId=302475739" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**GAN Genrator network**


In [1]:
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, latent_dim=100, out_channels=3, feature_dim=64):
        super().__init__()
        self.main = nn.Sequential(
            # input: latent_dim x 1 x 1
            nn.ConvTranspose2d(latent_dim, feature_dim * 8, 4, 1, 0, bias=False),
            nn.LayerNorm([feature_dim * 8, 4, 4]),
            nn.ReLU(True),
            # state: feature_dim*8 x 4 x 4
            nn.ConvTranspose2d(feature_dim * 8, feature_dim * 4, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim * 4, 8, 8]),
            nn.ReLU(True),
            # state: feature_dim*4 x 8 x 8
            nn.ConvTranspose2d(feature_dim * 4, feature_dim * 2, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim * 2, 16, 16]),
            nn.ReLU(True),
            # state: feature_dim*2 x 16 x 16
            nn.ConvTranspose2d(feature_dim * 2, feature_dim, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim, 32, 32]),
            nn.ReLU(True),
            # state: feature_dim x 32 x 32
            nn.ConvTranspose2d(feature_dim, out_channels, 3, 1, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

**GAN Discriminator network**

In [2]:
class Discriminator(nn.Module):
    def __init__(self, in_channels=3, feature_dim=64):
        super().__init__()
        self.main = nn.Sequential(
            # input: 3 x 64 x 64
            nn.Conv2d(in_channels, feature_dim, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state: feature_dim x 32 x 32
            nn.Conv2d(feature_dim, feature_dim * 2, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim * 2, 16, 16]),
            nn.LeakyReLU(0.2, inplace=True),
            # state: feature_dim*2 x 16 x 16
            nn.Conv2d(feature_dim * 2, feature_dim * 4, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim * 4, 8, 8]),
            nn.LeakyReLU(0.2, inplace=True),
            # state: feature_dim*4 x 8 x 8
            nn.Conv2d(feature_dim * 4, feature_dim * 8, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim * 8, 4, 4]),
            nn.LeakyReLU(0.2, inplace=True),
            # state: feature_dim*8 x 4 x 4
            nn.Conv2d(feature_dim * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x).view(-1, 1)

In [3]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

transform = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset = ImageFolder('dataset', transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
netG = Generator().to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=0.0011, betas=(0.3, 0.978))
optimizerG = optim.Adam(netG.parameters(), lr=0.0011, betas=(0.3, 0.978))

for epoch in range(1000):  # 1000 iterations (epochs if one batch per iteration)
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        batch_size = real_imgs.size(0)
        
        # Train Discriminator
        netD.zero_grad()
        label_real = torch.full((batch_size,), 1., device=device)
        label_fake = torch.full((batch_size,), 0., device=device)
        
        output = netD(real_imgs).view(-1)
        lossD_real = criterion(output, label_real)
        
        noise = torch.randn(batch_size, 100, 1, 1, device=device)
        fake_imgs = netG(noise)
        output = netD(fake_imgs.detach()).view(-1)
        lossD_fake = criterion(output, label_fake)
        
        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()
        
        # Train Generator
        netG.zero_grad()
        output = netD(fake_imgs).view(-1)
        lossG = criterion(output, label_real)  # fool D
        lossG.backward()
        optimizerG.step()
        
        print(f'Epoch [{epoch+1}/1000] Loss D: {lossD.item():.4f}, Loss G: {lossG.item():.4f}')

FileNotFoundError: [Errno 2] No such file or directory: 'dataset'

In [5]:
!git clone https://github.com/WongKinYiu/yolov7.git
%cd yolov7
!pip install -r requirements.txt

Cloning into 'yolov7'...
remote: Enumerating objects: 1197, done.
remote: Total 1197 (delta 0), reused 0 (delta 0), pack-reused 1197 (from 1)
Receiving objects: 100% (1197/1197), 74.29 MiB | 25.54 MiB/s, done.
Resolving deltas: 100% (513/513), done.
/kaggle/working/yolov7
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 55.8 MB/s eta 0:00:0000:010:01
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [6]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/train.py
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/LICENSE.md
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/hubconf.py
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/.gitignore
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/test.py
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/README.md
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/export.py
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/train_aux.py
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/requirements.txt
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/detect.py
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/utils/plots.py
/kaggle/input/models/mohosama/yolov7/pytorch/default/1/yolov7-main/utils/metrics.py
/kaggle/input/models/mohosama/yolov7/pytorch/default/1

In [7]:
!git clone https://github.com/WongKinYiu/yolov7
%cd yolov7

Cloning into 'yolov7'...
remote: Enumerating objects: 1197, done.
remote: Total 1197 (delta 0), reused 0 (delta 0), pack-reused 1197 (from 1)
Receiving objects: 100% (1197/1197), 74.29 MiB | 25.61 MiB/s, done.
Resolving deltas: 100% (511/511), done.
/kaggle/working/yolov7/yolov7


In [9]:
!pip install pyyaml tqdm matplotlib seaborn pandas opencv-python

In [8]:
import torch

model = torch.hub.load('.', 'custom', 
                       path='/kaggle/input/models/mohosama/yolov7/pytorch/default/1', 
                       source='local')
model.eval()

requirements: numpy<1.24.0,>=1.18.5 not found and is required by YOLOR, attempting auto-update...


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


CalledProcessError: Command 'pip install 'numpy<1.24.0,>=1.18.5'' returned non-zero exit status 1.

**GIoU Loss**

In [4]:
def giou_loss(pred_boxes, target_boxes):
    """
    pred_boxes, target_boxes: (N, 4) in (x1, y1, x2, y2) format
    """
    x1p, y1p, x2p, y2p = pred_boxes.unbind(-1)
    x1t, y1t, x2t, y2t = target_boxes.unbind(-1)
    
    # Intersection
    x1i = torch.max(x1p, x1t)
    y1i = torch.max(y1p, y1t)
    x2i = torch.min(x2p, x2t)
    y2i = torch.min(y2p, y2t)
    inter_area = (x2i - x1i).clamp(0) * (y2i - y1i).clamp(0)
    
    # Union
    area_p = (x2p - x1p) * (y2p - y1p)
    area_t = (x2t - x1t) * (y2t - y1t)
    union_area = area_p + area_t - inter_area
    
    iou = inter_area / (union_area + 1e-7)
    
    # Enclosing box
    x1c = torch.min(x1p, x1t)
    y1c = torch.min(y1p, y1t)
    x2c = torch.max(x2p, x2t)
    y2c = torch.max(y2p, y2t)
    area_c = (x2c - x1c) * (y2c - y1c)
    
    giou = iou - (area_c - union_area) / (area_c + 1e-7)
    loss = 1 - giou
    return loss.mean()